In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
!pip install pylint langchain langchain-text-splitters langchain-community llama_index langchain-google-genai google-generativeai python-dotenv

In [4]:
import os
from dotenv import load_dotenv
load_dotenv()
from llama_parse import LlamaParse

llama_parser_api_key = os.getenv("LLAMA_CLOUD_API_KEY")

parser = LlamaParse(
    api_key= llama_parser_api_key,
    result_type="markdown",
)
documents = parser.load_data("../data/clean_code_full.pdf")

with open("document.txt", "w", encoding="utf-8") as file:
    for doc in documents:
        file.write(doc.text)
        file.write("\n\n")

Started parsing the file under job_id b53b60a3-dab9-4f96-8774-4e0cda1a9d2d


In [5]:
from dotenv import load_dotenv
import os
load_dotenv()

from llama_index.core.query_engine import RouterQueryEngine
from llama_index.core import Settings
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding
from llama_index.core import Document
from llama_index.core import  VectorStoreIndex, SummaryIndex
from llama_index.core.tools import QueryEngineTool
from llama_index.core.selectors.utils import get_selector_from_llm

from llama_index.core.node_parser import SentenceSplitter

transformations = [
    SentenceSplitter(chunk_size=400, chunk_overlap=0)
]

api_key = os.getenv("GOOGLE_API_KEY")

Settings.llm = GoogleGenAI(model="gemini-2.5-flash-lite", api_key=api_key)
Settings.embed_model = GoogleGenAIEmbedding(model_name="gemini-embedding-001", api_key=api_key)

with open("document.txt", "r", encoding="utf-8") as f:
    text = f.read()

documents = [Document(text=text)]


vectorIndex=VectorStoreIndex.from_documents(documents)
summaryIndex=SummaryIndex.from_documents(documents)

vectorEngine=vectorIndex.as_query_engine(
    similarity_top_k=5,
    transformations=transformations
)


summaryEngine=summaryIndex.as_query_engine(
    responses_model='tree_summarize',
    use_async=True
)

vectorTool = QueryEngineTool.from_defaults(
    query_engine=vectorEngine,
    description=(
        "Use this for questions that need specific facts, quotes, definitions, "
        "or details from the document."
    ),
)
summaryTool = QueryEngineTool.from_defaults(
    query_engine=summaryEngine,
    description=(
        "Use this for summarization questions, high-level overviews, or "
        "when the user asks to summarize a chapter/section/document."
    ),
)

tools = [vectorTool, summaryTool]
selector=get_selector_from_llm(Settings.llm)

routerEngine=RouterQueryEngine(
    selector=selector,
    query_engine_tools=tools,
    verbose=True
)

# librarian_tool = QueryEngineTool.from_defaults(
#     query_engine=routerEngine,
#     name="clean_code_librarian",
#     description=(
#         "Use this tool to retrieve principles, quotes, and explanations "
#         "from Robert C. Martin's book 'Clean Code'. "
#         "Use it when you need justification or guidance about why a piece "
#         "of code is bad or violates clean coding practices."
#     )
# )


In [1]:
import json
import os
import subprocess
import tempfile
from langchain_core.tools import tool


@tool
def pylint_linter(code: str) -> dict:
    """
    Runs pylint on Python code returns two things
    1. number of lines in the function
    2. the presence of magic numbers
    """
    file_path = None

    try:
        with tempfile.NamedTemporaryFile(
            delete=False,
            suffix=".py",
            mode="w",
            encoding="utf-8"
        ) as f:
            f.write(code)
            file_path = f.name

        result = subprocess.run(
            [
                "pylint",
                file_path,
                "--output-format=json",
                "--load-plugins=pylint.extensions.magic_value",
                "--disable=all",
                "--enable=magic-value-comparison",
            ],
            capture_output=True,
            text=True,
            encoding="utf-8"
        )

        stdout = result.stdout.strip()
        issues = json.loads(stdout) if stdout else []

        return {
            "line_count": len(code.splitlines()),
            "magic_number_issues": issues,
        }

    except FileNotFoundError:
        return {"error": "pylint is not installed or not found in PATH"}
    except json.JSONDecodeError as e:
        return {"error": f"Invalid JSON from pylint: {e}"}
    except Exception as e:
        return {"error": str(e)}
    finally:
        if file_path and os.path.exists(file_path):
            os.remove(file_path)

In [18]:
code = """
def add(a,b):
    if x == 10:
        return 10
    result = a + b + 10
    return result
add(5, 10)
"""

print(pylint_linter.invoke(code))

{'line_count': 7, 'magic_number_issues': [{'type': 'refactor', 'module': 'tmpylm7anpg', 'obj': 'add', 'line': 3, 'column': 7, 'endLine': 3, 'endColumn': 14, 'path': 'C:\\Users\\pspra\\AppData\\Local\\Temp\\tmpylm7anpg.py', 'symbol': 'magic-value-comparison', 'message': "Consider using a named constant or an enum instead of '10'.", 'message-id': 'R2004'}]}


In [19]:
load_dotenv()
SYSTEM_PROMPT = """
You are a Code Auditor that evaluates Python code quality.

You have access to two tools:

1. code_linter:
Use this tool to analyze the given code snippet and it should focus on two things
    1. number of lines in the function
    2. the presence of magic numbers


2. clean_code_librarian:
Use this tool to retrieve principles or quotes from the book 'Clean Code'
by Robert C. Martin that explain why certain coding practices are bad.

Your workflow:

Step 1: Use the code_linter tool to analyze the code and identify measurable issues like heavy function defination(lines of code >50) or magical numbers presence . if you found multiple issues and consider all of them .
Step 2: Based on the issues found, use the clean_code_librarian tool to retrieve
relevant Clean Code principles.
Step 3: Produce a final explanation describing why the code is unclean,
using both the metrics from the linter and the Clean Code principle.

Your response should clearly explain:
- What the problem in the code is
- The metric detected by the linter
- The Clean Code principle that justifies the issue.

Be concise but clear in your explanation.
"""

from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
from langchain_core.tools import tool

@tool
def clean_code_librarian(query: str) -> str:
    """
    Retrieve Clean Code principles from the Clean Code book
    to justify why a code practice is unclean.
    """
    response = routerEngine.query(query)
    return str(response)

tools = [pylint_linter, clean_code_librarian]

agent = create_agent(
    llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT
)

response = agent.invoke({"messages": code}, verbose=True)

print(response)

Selecting query engine 0: The user is asking for a specific fact ('magic numbers') which requires retrieving details from the document..
{'messages': [HumanMessage(content='\ndef add(a,b):\n    if x == 10:\n        return 10\n    result = a + b + 10\n    return result\nadd(5, 10)\n', additional_kwargs={}, response_metadata={}, id='4757b01e-08bd-419b-9e47-b375e6af6bd4'), AIMessage(content='', additional_kwargs={'function_call': {'name': 'pylint_linter', 'arguments': '{"code": "def add(a,b):\\n    if x == 10:\\n        return 10\\n    result = a + b + 10\\n    return result"}'}, '__gemini_function_call_thought_signatures__': {'8b80184e-7367-4b2e-a31e-2178e76785d5': 'CrEEAb4+9vvxfpqI30z8S8hdjPVJ16X51q2ay/JSeyCE6fLZ59CbfWFPS3Rf0UIUiKZF5/oY+Dipr+hz5Cdhn7zDBt38qlZh2DSwTRqlVJfHfnK4ZyyXjJvhCekz9gFMtFXuz2MXE+OJFJ9WtPIC6N86vn6hHU6y4B5R752irTEL+60f/WJcZAe1p9PXpEbgE6Mc6H3uBJ4qeimQ+xg5Uu4MrVMOYHfSJ1cXnujw1Bxpecs4XB/ua0E+mfpAgFKgEr/niW/DJsggBJGIlw7TTxbSkQv/7w6G0r4uahQQ6/+gnlY0usJpKF2aY6EjUi5ABfeo38k

In [21]:
print(response["messages"][-1].content)

[{'type': 'text', 'text': 'The problem in the code is the presence of a "magic number".\n\nThe linter detected that the number `10` is used directly within the `add` function without being assigned to a descriptive constant.\n\nAccording to Clean Code principles, magic numbers are raw numbers in code that are not self-describing. It is generally a good practice to hide raw numbers behind descriptive names (named constants) to improve readability, make the code more understandable, and reduce the chance of errors.', 'extras': {'signature': 'CowEAb4+9vuy/dPdsyopxeLm9rejm/811zRiuuhTPMHzk/S+PZSoSI5+r+ZW+hfJx9Tde+WCbn8qcH/PfzaK2xwt2LK4UZFxWsKvbeBDLJsUCT/zmnTRnJUYXMQADOtjbLR0W0FJVNsi+kH8FVwcb4OV9dXc4tKVONJIFZ2Jm06uEsQdoX8Ml0/sGZx/glgjUIUZ14SpuJo0LQ5//H5RNq8GLQQZlfP+cm3T7jqb12vOrEJUqeCVaY1Y3h3GAfx4fQ822EDSQEkP4LJdmXUiqC1sQePeexpsWlPHXWIgeovK7WziyYMHPjcmwEv6IP8+o2y0ECAg3KSXlpZeWj+RekuY9ZvUlvWMz/7UNHKBKzLTHXeoMUf71PTDlSZhrucpSxittJ+YGEoTHdkolKwSd0Uwy5arAtDGnaPj4DA74KALsVvKlYpZtHIKViZ1slkYqr5mX2

In [22]:
for msg in response["messages"]:
    print(f"\n----- {msg.__class__.__name__} -----")

    if hasattr(msg, "content"):
        print(msg.content)

    if hasattr(msg, "tool_calls") and msg.tool_calls:
        print("Tool Calls:", msg.tool_calls)


----- HumanMessage -----

def add(a,b):
    if x == 10:
        return 10
    result = a + b + 10
    return result
add(5, 10)


----- AIMessage -----

Tool Calls: [{'name': 'pylint_linter', 'args': {'code': 'def add(a,b):\n    if x == 10:\n        return 10\n    result = a + b + 10\n    return result'}, 'id': '8b80184e-7367-4b2e-a31e-2178e76785d5', 'type': 'tool_call'}]

----- ToolMessage -----
{"line_count": 5, "magic_number_issues": [{"type": "refactor", "module": "tmp2ajhy5yh", "obj": "add", "line": 2, "column": 7, "endLine": 2, "endColumn": 14, "path": "C:\\Users\\pspra\\AppData\\Local\\Temp\\tmp2ajhy5yh.py", "symbol": "magic-value-comparison", "message": "Consider using a named constant or an enum instead of '10'.", "message-id": "R2004"}]}

----- AIMessage -----

Tool Calls: [{'name': 'clean_code_librarian', 'args': {'query': 'magic numbers'}, 'id': '75304c7e-d341-4341-b102-cda01b4fc9e5', 'type': 'tool_call'}]

----- ToolMessage -----
Magic numbers are raw numbers in code that 

In [13]:
import json

print(json.dumps(response, indent=2, default=str))

{
  "messages": [
    "content='\\ndef add(a,b):\\n    if x == 10:\\n        return 10\\n    result = a + b + 10\\n    result = a + b + 10\\n    result = a + b + 10\\n    result = a + b + 10\\n    result = a + b + 10\\n    result = a + b + 10\\n    result = a + b + 10\\n    result = a + b + 10\\n    result = a + b + 10\\n    result = a + b + 10\\n    result = a + b + 10\\n    result = a + b + 10\\n    result = a + b + 10\\n    result = a + b + 10\\n    result = a + b + 10\\n    result = a + b + 10\\n    result = a + b + 10\\n    result = a + b + 10\\n    result = a + b + 10\\n    result = a + b + 10\\n    result = a + b + 10\\n    result = a + b + 10\\n    result = a + b + 10\\n    result = a + b + 10\\n    result = a + b + 10\\n    result = a + b + 10\\n    result = a + b + 10\\n    result = a + b + 10\\n    result = a + b + 10\\n    result = a + b + 10\\n    result = a + b + 10\\n    result = a + b + 10\\n    result = a + b + 10\\n    result = a + b + 10\\n    result = a + b + 10\\n 

In [14]:
print(response["messages"][-1].content)

[{'type': 'text', 'text': 'The provided code is unclean due to two main issues: excessive function length and the use of magic numbers.\n\n**Problem:** The `add` function is overly long and contains a hardcoded, unexplained numerical value.\n\n**Metrics detected by the linter:**\n*   **Function Length:** The `add` function spans 58 lines, significantly exceeding recommended function lengths.\n*   **Magic Number:** The number `10` is used directly in the code (e.g., `if x == 10:` and `result = a + b + 10`) without any explanation or assignment to a named constant.\n\n**Clean Code Principle:**\n*   **Function Length:** According to Clean Code principles, "Functions should be very small. A common guideline is that functions should not be longer than a screen-full, which historically meant around 20 lines. Ideally, functions should be only two to four lines long, making them transparently obvious and easy to understand." The current 58-line function violates this principle, making it harde